# SBERT 기반 Hierarchical Alignment(계층 정렬)

In [1]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ======================
# 경로/파라미터
# ======================
JOB_PATH = "datas/processed/job_postings_processed.csv"     # job_posting_id, text
NCS_PATH = "datas/processed/ncs_roles_processed.csv"       # role_text, job_role_name, (가능하면) large/mid/small
OUT_PATH = "outputs/sbert_hier_top3.csv"

MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"
BATCH_SIZE = 32

K_SMALL = 1     # 1단계에서 소분류 Top-몇개까지 후보로 둘지 (보통 1~3)
TOP_K  = 3      # 2단계에서 세분류 Top-k

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ======================
# 0) 로드
# ======================
job = pd.read_csv(JOB_PATH, dtype={"job_posting_id": "string"})
ncs = pd.read_csv(NCS_PATH)

# 필수 컬럼 체크
if "job_posting_id" not in job.columns or "text" not in job.columns:
    raise ValueError("job 파일에 job_posting_id, text 컬럼이 필요합니다.")
if "role_text" not in ncs.columns:
    raise ValueError("ncs 파일에 role_text 컬럼이 필요합니다.")
if "job_role_name" not in ncs.columns:
    raise ValueError("ncs 파일에 job_role_name 컬럼이 필요합니다.")

# 계층 컬럼 자동 탐지 (있으면 사용)
# - 소분류 = small_category 를 우선 사용
# - 없으면 sub_id 같은 컬럼으로 대체 불가(계층 정렬 자체가 성립이 어려움)
SMALL_COL = "small_category" if "small_category" in ncs.columns else None
MID_COL   = "mid_category"   if "mid_category"   in ncs.columns else None
LARGE_COL = "large_category" if "large_category" in ncs.columns else None

if SMALL_COL is None:
    raise ValueError(
        "Hierarchical Alignment(1단계 소분류 정렬)을 하려면 ncs에 'small_category' 컬럼이 필요합니다.\n"
        "현재 파일에는 small_category가 없습니다. ncs 전처리에서 소분류 컬럼을 추가해 주세요."
    )

# ID 컬럼(있으면 결과에 포함)
ID_COL = None
for cand in ["ncs_id", "job_role_id"]:
    if cand in ncs.columns:
        ID_COL = cand
        break

# 정리
job["job_posting_id"] = job["job_posting_id"].astype(str).str.strip()
job["text"] = job["text"].fillna("").astype(str)

ncs["job_role_name"] = ncs["job_role_name"].astype(str).str.strip()
ncs["role_text"] = ncs["role_text"].fillna("").astype(str)
ncs[SMALL_COL] = ncs[SMALL_COL].astype(str).str.strip()

# ======================
# 1) 소분류 대표 텍스트 만들기
#   - 같은 소분류에 속한 role_text들을 합쳐서 "소분류 프로필" 생성
# ======================
small_profiles = (
    ncs.groupby(SMALL_COL)["role_text"]
       .apply(lambda s: " ".join([x for x in s.tolist() if isinstance(x, str) and x.strip()]))
       .reset_index()
       .rename(columns={"role_text": "small_profile_text"})
)

small_labels = small_profiles[SMALL_COL].tolist()
small_texts  = small_profiles["small_profile_text"].tolist()

# ======================
# 2) 임베딩 생성
# ======================
model = SentenceTransformer(MODEL_NAME)

# 소분류 프로필 임베딩
small_emb = model.encode(
    small_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 공고 임베딩
post_texts = job["text"].tolist()
post_emb = model.encode(
    post_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# ======================
# 3) 1단계: 소분류 Top-K_SMALL 예측
# ======================
sim_small = post_emb @ small_emb.T  # (num_posts, num_small)

# Top-K_SMALL 인덱스
top_small_idx = np.argpartition(-sim_small, K_SMALL - 1, axis=1)[:, :K_SMALL]
top_small_scores = np.take_along_axis(sim_small, top_small_idx, axis=1)

# 정렬(내림차순)
order = np.argsort(-top_small_scores, axis=1)
top_small_idx = np.take_along_axis(top_small_idx, order, axis=1)
top_small_scores = np.take_along_axis(top_small_scores, order, axis=1)

# ======================
# 4) 2단계: (예측 소분류 내부) 세분류 Top-K
# ======================
rows = []

# ncs를 소분류별로 묶어 두면 빠름
ncs_by_small = {k: v.reset_index(drop=True) for k, v in ncs.groupby(SMALL_COL)}

for i, post_id in enumerate(job["job_posting_id"].tolist()):
    # 1단계 소분류 후보(Top-K_SMALL)
    for s_rank in range(K_SMALL):
        s_idx = int(top_small_idx[i, s_rank])
        pred_small = small_labels[s_idx]
        pred_small_score = float(top_small_scores[i, s_rank])

        cand_ncs = ncs_by_small.get(pred_small)
        if cand_ncs is None or len(cand_ncs) == 0:
            continue

        cand_texts = cand_ncs["role_text"].tolist()

        # 후보 세분류 임베딩: 소분류별로 매번 encode하면 느릴 수 있음.
        # 파일럿/작은 규모에서는 OK. (규모 커지면 캐싱 권장)
        cand_emb = model.encode(
            cand_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        # 공고 i 와 후보 세분류 간 유사도
        sim_role = post_emb[i:i+1] @ cand_emb.T  # (1, num_roles_in_small)
        sim_role = sim_role.flatten()

        # Top-K
        k = min(TOP_K, len(sim_role))
        top_idx = np.argpartition(-sim_role, k - 1)[:k]
        top_scores = sim_role[top_idx]
        ord2 = np.argsort(-top_scores)
        top_idx = top_idx[ord2]
        top_scores = top_scores[ord2]

        # 결과 저장(행: post x rank)
        for r, (j, score) in enumerate(zip(top_idx, top_scores), start=1):
            out = {
                "job_posting_id": post_id,
                "small_pred_rank": s_rank + 1,
                "pred_small_category": pred_small,
                "pred_small_score": pred_small_score,
                "rank": r,
                "score": float(score),
                "job_role_name": cand_ncs.loc[int(j), "job_role_name"],
            }
            if ID_COL:
                out["job_role_id"] = str(cand_ncs.loc[int(j), ID_COL]).strip()

            # (있으면) 상위 계층도 같이 첨부
            if LARGE_COL:
                out[LARGE_COL] = cand_ncs.loc[int(j), LARGE_COL]
            if MID_COL:
                out[MID_COL] = cand_ncs.loc[int(j), MID_COL]
            out[SMALL_COL] = cand_ncs.loc[int(j), SMALL_COL]

            rows.append(out)

hier_topk = pd.DataFrame(rows)

# ======================
# 5) 저장
# ======================
hier_topk.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[OK] saved: {OUT_PATH}  | rows={len(hier_topk)}")
print(f"[INFO] posts={len(job)} | K_SMALL={K_SMALL} | TOP_K={TOP_K}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[OK] saved: outputs/sbert_hier_top3.csv  | rows=324
[INFO] posts=108 | K_SMALL=1 | TOP_K=3
